En este notebook se ejecuta el Análisis Exploratorio de los Datos.
Se genera un reporte sobre lo encontrado y sobre las correcciones que es necesario efectuar.

In [2]:
from _bootstrap import setup
from src.vasijas.reporting import ProcessingReport

In [3]:
PATHS = setup()
report = ProcessingReport()

In [4]:
# Ubicaciones detectadas en los datos originales

ubicaciones = [item.name for item in PATHS.DATA_ORIGINALES.iterdir() if item.is_dir()]

report.add_metric("cantidad_ubicaciones encontradas", len(ubicaciones))
report.add_metric("ubicaciones_detectadas", ubicaciones)

print(f"Cantidad de ubicaciones: {len(ubicaciones)}\n")
for ubicacion in ubicaciones:
    print(ubicacion)

Cantidad de ubicaciones: 22

Almedinilla
Atalayuelas
Baños de la Muela
Baza
Castellones de Ceal
Cazalilla
Cerrillo de los Gordos
Cerro de la Compañía
Estacar de Luciano
Estacar de Robarinas
Giribaile
La Bobadilla
La Guardia
Loma del Peinado
Los Patos
Marroquies Bajos
Molino de Caldona
Puente del Obispo
Puente Tablas
Puerta Norte
Toya
Tútugi


In [5]:
# Perfiles encontrados por ubicación

total_png = 0
conteo_png: dict[str, int] = dict()

for ruta_ubicacion in PATHS.DATA_ORIGINALES.iterdir():
    if ruta_ubicacion.is_dir():
        pngs = list(ruta_ubicacion.glob("*.png"))
        conteo_png[ruta_ubicacion.name] = len(pngs)
        total_png += len(pngs)

print(f"Total de archivos (perfiles) en filesystem: {total_png}\n")
for ubicacion, conteo in conteo_png.items():
    print(f"{ubicacion}: {conteo}")

report.add_metric("total_perfiles", total_png)
report.add_metric("perfiles_por_ubicacion", conteo_png)

Total de archivos (perfiles) en filesystem: 1242

Almedinilla: 66
Atalayuelas: 35
Baños de la Muela: 6
Baza: 368
Castellones de Ceal: 182
Cazalilla: 12
Cerrillo de los Gordos: 30
Cerro de la Compañía: 4
Estacar de Luciano: 16
Estacar de Robarinas: 27
Giribaile: 7
La Bobadilla: 51
La Guardia: 70
Loma del Peinado: 43
Los Patos: 16
Marroquies Bajos: 18
Molino de Caldona: 3
Puente del Obispo: 28
Puente Tablas: 4
Puerta Norte: 135
Toya: 21
Tútugi: 100


In [6]:
# Comprobación de concordancia entre metadata.csv y archivos de perfiles

# Dataframe
import pandas as pd

archivo_csv = PATHS.DATA_ORIGINALES / "metadata.csv"
df = pd.read_csv(archivo_csv)

df.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia
0,AL_03A,No,II,1,B,2,2,11.0,182,54,53,ibero
1,AL_03B,No,II,1,B,2,2,11.0,178,57,46,ibero
2,AL_03C,No,II,1,B,2,2,11.0,185,56,49,ibero
3,AL_03D,No,II,1,B,2,2,11.0,160,41,45,ibero
4,AL_03E,No,II,1,B,2,2,11.0,164,54,44,ibero


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1242 entries, 0 to 1241
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Identificador     1242 non-null   str    
 1   Tiene asa         1242 non-null   str    
 2   Tendencia Formal  1242 non-null   str    
 3   Borde             1242 non-null   int64  
 4   Cuello            1242 non-null   str    
 5   Cuerpo            1242 non-null   int64  
 6   Base              1242 non-null   int64  
 7   Clase Ana         1081 non-null   float64
 8   Diametro Borde    1242 non-null   int64  
 9   Diametro Base     1242 non-null   int64  
 10  Altura Total      1242 non-null   int64  
 11  Cronologia        1241 non-null   str    
dtypes: float64(1), int64(6), str(5)
memory usage: 116.6 KB


In [12]:
# Archivos
nombres_archivos_png: set[str] = {p.stem for p in PATHS.DATA_ORIGINALES.rglob("*.png")}

total_png_rglob = sum(1 for _ in PATHS.DATA_ORIGINALES.rglob("*.png"))

print(f"Total de archivos PNG encontrados: {total_png_rglob}")
# Preview
print(f"Muestra:{list(nombres_archivos_png)[:10]}")

Total de archivos PNG encontrados: 1242
Muestra:['BA_137_3', 'CC_06_7', 'PN_4_2', 'BA_11_2', 'AL_03I', 'AL_03B', 'CC_21_3', 'PN_23_2', 'LP_11_2', 'EL_01_7']


In [9]:
# Identificadores del dataset que aparecen en los nombres de los archivos
df["id_encontrado_en_fs"] = (
    df["Identificador"]
    .astype(str)
    .str.strip()
    .isin(nombres_archivos_png)
)

cantidad_encontrados = int(df["id_encontrado_en_fs"].sum())
cantidad_no_encontrados = int((~df["id_encontrado_en_fs"]).sum())

report.add_metric("identificadores_encontrado_en_filesystem", cantidad_encontrados)
report.add_metric("identificadores_no_encontrado_en_filesystem", cantidad_no_encontrados)

print(f"\nCantidad de encontrados: {cantidad_encontrados}")
print(f"Cantidad de no encontrados: {cantidad_no_encontrados}\n")


Cantidad de encontrados: 1177
Cantidad de no encontrados: 65



In [10]:
# Detalle de Identificadores del dataset que no encontrados en el conjunto de nombres de archivos:

no_encontrados = df.loc[~df["id_encontrado_en_fs"], "Identificador"].astype(str).str.strip()
print("Identificadores en el dataset no encontrados en el conjunto de nombres de archivos:")
print(no_encontrados.to_string(index=False))

Identificadores en el dataset no encontrados en el conjunto de nombres de archivos:
      Al_03G
      Al_06A
     BA115_5
     BA132_1
      BO_3_1
      BO_4_1
      BO_4_2
      BO_4_3
      BO_5_1
 BO_5_1plato
      BO_6_1
      BO_6_2
      BO_7_1
      BO_7_2
      BO_9_1
      BO_9_2
      BO_9_3
      BO_9_4
     BO_10_1
BO_10_1plato
    BO_11a_1
    BO_11a_2
    BO_11a_3
    BO_11b_1
     BO_14_1
     BO_14_2
     BO_16_1
     BO_17_1
     BO_18_1
     BO_18_2
     BO_19_1
     BO_19_2
     BO_20_1
     BO_21_1
     BO_21_2
     BO_22_1
     BO_23_1
     BO_24_1
     BO_26_1
     BO_27_1
     BO_27_2
     BO_28_1
     BO_28_2
     BO_29_1
     BO_29_2
     BO_29_3
     BO_29_4
     BO_29_5
     BO_30_1
     BO_30_2
     BO_31_1
     BO_31_2
     BO_31_3
     BO_31_4
     BO3_2_1
     CC0_8_2
    CC_11_14
    CC_11_15
    CC_11_17
    CC_11_18
CC_13_14_1_3
CC_13_14_2_1
CC_13_14_2_3
     ER_23_1
      PN10_1


In [14]:
# Identificadores presentes en el DataFrame
ids_en_df = set(df["Identificador"].astype(str).str.strip())

# Archivos PNG cuyos nombres NO están en el DataFrame
archivos_sin_registro = nombres_archivos_png - ids_en_df

report.add_metric("archivos_png_sin_registro_en_metadata", len(archivos_sin_registro))

print(f"Cantidad de archivos PNG sin registro en metadata: {len(archivos_sin_registro)}")
print("Detalle de archivos PNG no encontrados en metadata.csv:")
for nombre in sorted(archivos_sin_registro):
    print(f"  {nombre}")

Cantidad de archivos PNG sin registro en metadata: 65
Detalle de archivos PNG no encontrados en metadata.csv:
  AL_03G
  AL_06A
  BA_115_5
  BA_132_1
  BO10_1
  BO10_1plato
  BO11a_1
  BO11a_2
  BO11a_3
  BO11b_1
  BO14_1
  BO14_2
  BO16_1
  BO17_1
  BO18_1
  BO18_2
  BO19_1
  BO19_2
  BO20_1
  BO21_1
  BO21_2
  BO22_1
  BO23_1
  BO24_1
  BO26_1
  BO27_1
  BO27_2
  BO28_1
  BO28_2
  BO29_1
  BO29_2
  BO29_3
  BO29_4
  BO29_5
  BO30_1
  BO30_2
  BO31_1
  BO31_2
  BO31_3
  BO31_4
  BO32_1
  BO3_1
  BO4_1
  BO4_2
  BO4_3
  BO5_1
  BO5_1plato
  BO6_1
  BO6_2
  BO7_1
  BO7_2
  BO9_1
  BO9_2
  BO9_3
  BO9_4
  CC_08_2
  CC_11_ 14
  CC_11_ 15
  CC_11_ 17
  CC_11_ 18
  CC_13_14 1_3
  CC_13_14 2_1
  CC_13_14 2_3
  ER_23-1
  PN_10_1


In [11]:
report.save_markdown(PATHS.REPORTS / "processing_report.md")